# 🎓 Lab Tutorial: Custom Schema GraphRAG with Qdrant Vector Store (LlamaIndex + Ollama)

This notebook extends `2.graphrag_custom_schema_tutorial.ipynb` by replacing LlamaIndex's default in-memory `SimpleVectorStore` (RAM) with a persistent **Qdrant** vector database running in **Docker** (`http://localhost:6333`).

### 📌 What Changes When Using Qdrant?
- **Before (RAM):** Embeddings for document chunks and extracted graph nodes lived in Python memory and were lost when the Jupyter kernel restarted.
- **Now (Qdrant Docker):** All embeddings are stored in a Qdrant collection (`graphrag_schema_nodes`), inspectable in the Qdrant Web UI at `http://localhost:6333/dashboard`, and reusable across sessions via `PropertyGraphIndex.from_existing(...)`.

> **Prerequisite — Make sure your Qdrant Docker container is running:**
> ```bash
> docker run -d --name qdrant -p 6333:6333 -p 6334:6334 \
>   -v $(pwd)/qdrant_storage:/qdrant/storage qdrant/qdrant
> ```

---

## 1. Install Qdrant Integration (if not already installed)

In [ ]:
# Uncomment and run once if llama-index-vector-stores-qdrant is not installed yet:
# %pip install llama-index-vector-stores-qdrant qdrant-client

Searched "PropertyGraphIndex" "QdrantVectorStore" vector_store llama_index from_documents from_existing
Searched "VectorContextRetriever" "vector_store" "EntityNode" "node.id" llama_index property_graph
Created 3.graphrag_custom_schema_qdrant.ipynb
Ran validate Qdrant notebook JSON and syntax

I have created a new notebook using **Qdrant (Docker)** as the persistent vector store instead of RAM:

- [3.graphrag_custom_schema_qdrant.ipynb](file:///home/ashok/Documents/graph_rag/3.graphrag_custom_schema_qdrant.ipynb)

### Key Changes from [2.graphrag_custom_schema_tutorial.ipynb](file:///home/ashok/Documents/graph_rag/2.graphrag_custom_schema_tutorial.ipynb)
1. **Qdrant Docker Connection (Section 3):**
   - Connects via `qdrant_client.QdrantClient(url="http://localhost:6333")` and initializes `QdrantVectorStore(client=client, collection_name="graphrag_schema_nodes")`.
   - Includes a `RESET_COLLECTION = True` toggle so re-running the notebook doesn't create duplicate vectors.
2. **Schema-Guided Indexing into Qdrant (Section 4):**
   - Passes `vector_store=qdrant_vector_store` to `PropertyGraphIndex.from_documents(...)` so all document chunk and extracted entity embeddings are stored in Qdrant rather than in-memory `SimpleVectorStore`.
   - Prints `client.count(collection_name=COLLECTION_NAME).count` to verify points were written to Qdrant (also viewable at `http://localhost:6333/dashboard`).
3. **Persist & Reload Without Re-Extraction (Section 4.4):**
   - Saves the property graph JSON to `./storage_qdrant/property_graph_store.json` and demonstrates reloading both the graph (from disk) and vectors (from Qdrant) using `PropertyGraphIndex.from_existing(..., vector_store=reloaded_vector_store, use_async=False)`.
4. **Fixes Included:**
   - Uses `.graph.get_triplets()` for triplet inspection and `use_async=False` across all index/query calls for Jupyter compatibility.

*(If `llama-index-vector-stores-qdrant` isn't installed in your Python environment yet, uncomment and run the `%pip install` cell at the top of the notebook.)*

## 2. Setup & Environment Configuration
Configure `nest_asyncio`, local **Ollama** models (`qwen2.5:latest` with `temperature=0.0` + `nomic-embed-text:latest`), and load the 4 documents from `data/`.

In [ ]:
# 1.0
# CRITICAL FOR JUPYTER NOTEBOOKS:
import nest_asyncio
nest_asyncio.apply()

# 1.1
from pathlib import Path
from llama_index.core import Settings, SimpleDirectoryReader, PropertyGraphIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# 1.2 Paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
STORAGE_DIR = PROJECT_ROOT / "storage_qdrant"
STORAGE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# 1.3 Configure Local Ollama LLM (temperature=0.0 for reliable schema extraction)
llm = Ollama(
    model="qwen2.5:latest",
    request_timeout=800.0,
    temperature=0.0,
)

# 1.4 Global Embedding Model
embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

# 1.5
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

In [ ]:
# 1.6 Load the 4 documents from data/
reader = SimpleDirectoryReader(input_dir=str(DATA_DIR))
documents = reader.load_data()
print(f"Loaded {len(documents)} documents from {DATA_DIR}")

## 3. Connect to Qdrant Docker & Initialize `QdrantVectorStore`
Here we connect to the local Qdrant Docker instance on port `6333`. If you re-run this notebook from scratch, setting `RESET_COLLECTION = True` clears the old collection so duplicate vectors are not appended.

In [ ]:
# 2.0
import qdrant_client
from llama_index.vector_stores.qdrant import QdrantVectorStore

# 2.1
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "graphrag_schema_nodes"
RESET_COLLECTION = True  # Set to False if you want to keep existing points

# 2.2 Connect to Qdrant Docker
client = qdrant_client.QdrantClient(url=QDRANT_URL)

# 2.3 Verify connection
existing_collections = [c.name for c in client.get_collections().collections]
print(f"Connected to Qdrant at {QDRANT_URL}")
print(f"Existing collections: {existing_collections}")

# 2.4 Delete existing collection
if RESET_COLLECTION and COLLECTION_NAME in existing_collections:
    # 2.4.1
    client.delete_collection(collection_name=COLLECTION_NAME)
    # 2.4.2
    print(f"Deleted existing collection '{COLLECTION_NAME}' for a clean run.")

# 2.5 Create the LlamaIndex QdrantVectorStore
qdrant_vector_store = QdrantVectorStore(
                                        client=client,
                                        collection_name=COLLECTION_NAME,
                                        )
print(f"QdrantVectorStore ready (collection='{COLLECTION_NAME}').")

---
## 4. Approach 1: Schema-Guided Extraction (`SchemaLLMPathExtractor`) + Qdrant Vector Store

The only addition to `PropertyGraphIndex.from_documents(...)` is passing **`vector_store=qdrant_vector_store`**. LlamaIndex will automatically store all chunk and entity embeddings in Qdrant instead of RAM.

In [ ]:
# 3.0
from typing import Literal
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor

# 3.1 Define allowed Node Types (Entities)
entities = Literal[
                    "PERSON",
                    "COMPANY",
                    "INITIATIVE",
                    "TECHNOLOGY",
                    "LOCATION",
                    "REGULATOR",
                    ]

# 3.2 Define allowed Relationship Types (Edges)
relations = Literal[
                    "CO_FOUNDED",
                    "WORKS_AT",
                    "ADVISES",
                    "SERVES_ON_BOARD_OF",
                    "COLLABORATED_WITH",
                    "INVESTED_IN",
                    "FUNDED",
                    "JOINT_VENTURE_WITH",
                    "LICENSES_TO",
                    "DEVELOPS",
                    "TESTBED_FOR",
                    "HEADQUARTERED_IN",
                    "INVESTIGATES",
                    ]

# 3.3 Define valid (SourceEntityType, Relation, TargetEntityType) rules
validation_schema = [
                    ("PERSON", "CO_FOUNDED", "COMPANY"),
                    ("PERSON", "WORKS_AT", "COMPANY"),
                    ("PERSON", "ADVISES", "COMPANY"),
                    ("PERSON", "SERVES_ON_BOARD_OF", "COMPANY"),
                    ("PERSON", "COLLABORATED_WITH", "PERSON"),
                    ("PERSON", "DEVELOPS", "TECHNOLOGY"),
                    ("COMPANY", "INVESTED_IN", "COMPANY"),
                    ("COMPANY", "FUNDED", "INITIATIVE"),
                    ("COMPANY", "JOINT_VENTURE_WITH", "COMPANY"),
                    ("COMPANY", "LICENSES_TO", "COMPANY"),
                    ("COMPANY", "DEVELOPS", "TECHNOLOGY"),
                    ("COMPANY", "TESTBED_FOR", "INITIATIVE"),
                    ("COMPANY", "HEADQUARTERED_IN", "LOCATION"),
                    ("REGULATOR", "INVESTIGATES", "PERSON"),
                    ("REGULATOR", "INVESTIGATES", "COMPANY"),
                ]



In [ ]:
# 3.4 Create the Schema-Constrained Extractor
kg_extractor = SchemaLLMPathExtractor(
    llm=llm,
    possible_entities=entities,
    possible_relations=relations,
    kg_validation_schema=validation_schema,
    strict=True,
    max_triplets_per_chunk=10,
    num_workers=1,
)

In [ ]:
# 3.5
print("Constructing Schema-Guided Property Graph Index with Qdrant Vector Store...")
prop_index_schema = PropertyGraphIndex.from_documents(
                                                        documents,
                                                        kg_extractors=[kg_extractor],
                                                        vector_store=qdrant_vector_store,   # <-- Persist vectors in Qdrant instead of RAM
                                                        use_async=False,
                                                        show_progress=True,
                                                        


# 3.6 Verify vectors were stored in Qdrant
point_count = client.count(collection_name=COLLECTION_NAME).count
print(f"\nSchema-guided graph index built successfully!")
print(f"Vectors stored in Qdrant collection '{COLLECTION_NAME}': {point_count}")
print(f"Inspect them in your browser at: {QDRANT_URL}/dashboard")

### 4.1 Inspect the Extracted Entity Triplets

In [ ]:
# 4.0
from llama_index.core.graph_stores.types import EntityNode

# 4.1
all_triplets = index_schema.property_graph_store.graph.get_triplets()

# 4.2
entity_triplets = [
                    (subj, rel, obj)
                    for subj, rel, obj in all_triplets
                    if isinstance(subj, EntityNode) and isinstance(obj, EntityNode)
                  ]

# 4.3
print(f"Total extracted entity triplets: {len(entity_triplets)} (out of {len(all_triplets)} total graph edges)\n")

# 4.4
for subj, rel, obj in entity_triplets:
    # 4.4.1
    print(f"({subj.name} [{subj.label}]) --[{rel.label}]--> ({obj.name} [{obj.label}])")

### 4.2 Visualize the Schema-Guided Graph

In [ ]:
# 5.0
from IPython.display import IFrame

# 5.1
index_schema.property_graph_store.save_networkx_graph(name="./graph_schema_qdrant.html")

# 5.2
IFrame(src="./graph_schema_qdrant.html", width="100%", height="700px")

### 4.3 Query the Graph (Hybrid Qdrant Vector Search + Graph Traversal)

In [ ]:
# 6.0
query_text = "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"

# 6.1
schema_query_engine = prop_index_schema.as_query_engine(
                                                        include_text=True,
                                                        similarity_top_k=3,
                                                        use_async=False,
                                                        )

# 6.2
response_schema = schema_query_engine.query(query_text)

# 6.3
print("=== 💡 SCHEMA-GUIDED + QDRANT GRAPHRAG RESPONSE ===")
print(response_schema.response)

### 4.4 Persist Graph to Disk & Reload Later from Qdrant (No Re-Extraction!)
Because the vectors are already persisted inside Qdrant Docker, you only need to persist the graph structure JSON to disk. Next time you open the notebook, you can reload everything instantly with `PropertyGraphIndex.from_existing(...)` without calling Ollama to re-extract entities.

In [ ]:
# 7.0
from llama_index.core.graph_stores import SimplePropertyGraphStore

# 7.1 Save the property graph store to disk
graph_persist_path = STORAGE_DIR / "property_graph_store.json"
index_schema.property_graph_store.persist(persist_path=str(graph_persist_path))
# 7.1.1
print(f"Saved property graph store to: {graph_persist_path}")

In [ ]:
# 8.0 Reload both the graph store (from disk) and vectors (from Qdrant Docker)
reloaded_graph_store = SimplePropertyGraphStore.from_persist_path(str(graph_persist_path))
reloaded_vector_store = QdrantVectorStore(client=client, collection_name=COLLECTION_NAME)

# 8.1
reloaded_index = PropertyGraphIndex.from_existing(
    property_graph_store=reloaded_graph_store,
    vector_store=reloaded_vector_store,
    llm=llm,
    embed_model=embed_model,
    use_async=False,
)

In [ ]:
%%time

# 9.0
from llama_index.core.indices.property_graph import VectorContextRetriever

# 9.1
vector_retriever = VectorContextRetriever(
    reloaded_index.property_graph_store,
    vector_store=reloaded_index.vector_store,
    embed_model=embed_model,
    similarity_top_k=2,
    path_depth=1,
    include_text=True,  # Set False for even faster graph-triplets-only answers
)

# 9.2
reloaded_engine = reloaded_index.as_query_engine(
    sub_retrievers=[vector_retriever],
    use_async=False,
)

# 9.3
reloaded_response = reloaded_engine.query(
    "Who funded Project Nexus and which company serves as its testbed?"
)

In [ ]:
# 10.0 Query the reloaded index
#reloaded_engine = reloaded_index.as_query_engine(include_text=True, similarity_top_k=3, use_async=False)
#reloaded_response = reloaded_engine.query("Who funded Project Nexus and which company serves as its testbed?")

# 10.1
print("\n=== ♻️ RELOADED INDEX RESPONSE (FROM QDRANT + DISK) ===")
print(reloaded_response.response)

---
## 5. Approach 2: Manually Defining Exact Nodes & Relationships

When you want 100% deterministic control over every node and edge (with zero LLM extraction), construct `EntityNode` and `Relation` objects directly and wrap the store with `PropertyGraphIndex.from_existing(..., use_async=False)`.

In [ ]:
# 11.0
from llama_index.core.graph_stores import (
    SimplePropertyGraphStore,
    EntityNode,
    Relation,
)

# 11.1
manual_store = SimplePropertyGraphStore()

# 11.2 Explicitly define Nodes
elena = EntityNode(
    name="Dr. Elena Vance",
    label="PERSON",
    properties={"role": "Co-Founder & Chief Scientific Advisor", "specialty": "CryoQubit & Quantum Compression"},
)
marcus = EntityNode(
    name="Marcus Thorne",
    label="PERSON",
    properties={"role": "Co-Founder & CEO"},
)
kenji = EntityNode(
    name="Dr. Kenji Sato",
    label="PERSON",
    properties={"role": "Founder & CEO", "background": "Former MIT robotics faculty"},
)
julian = EntityNode(
    name="Julian Mercer",
    label="PERSON",
    properties={"role": "Principal Investment Director"},
)
aurora = EntityNode(
    name="Aurora Technologies",
    label="COMPANY",
    properties={"headquarters": "Seattle, WA", "founded": 2021, "focus": "Quantum computing hardware"},
)
helios = EntityNode(
    name="Helios Capital",
    label="COMPANY",
    properties={"headquarters": "New York City", "fund": "Frontier Tech Fund IV ($1.2B)"},
)
aether = EntityNode(
    name="Aether Labs",
    label="COMPANY",
    properties={"type": "Joint Venture", "established": "September 2023"},
)
synapse = EntityNode(
    name="Synapse Dynamics",
    label="COMPANY",
    properties={"headquarters": "Boston", "founded": 2019, "focus": "Biomimetic humanoid robots"},
)
nexus = EntityNode(
    name="Project Nexus",
    label="INITIATIVE",
    properties={"status": "Stealth consortium", "goal": "Quantum-powered autonomous robotics"},
)

# 11.2.1
nodes = [elena, marcus, kenji, julian, aurora, helios, aether, synapse, nexus]

# 11.3 Explicitly define Relationships
relations = [
    Relation(label="CO_FOUNDED", source_id=elena.id, target_id=aurora.id, properties={"year": 2021}),
    Relation(label="CO_FOUNDED", source_id=marcus.id, target_id=aurora.id, properties={"year": 2021}),
    Relation(label="ADVISES", source_id=elena.id, target_id=synapse.id, properties={"since": "March 2024", "title": "Chief Scientific Advisor"}),
    Relation(label="COLLABORATED_WITH", source_id=elena.id, target_id=kenji.id, properties={"location": "Stanford University", "year": 2016}),
    Relation(label="CO_FOUNDED", source_id=kenji.id, target_id=synapse.id, properties={"year": 2019}),
    Relation(label="WORKS_AT", source_id=julian.id, target_id=helios.id),
    Relation(label="SERVES_ON_BOARD_OF", source_id=julian.id, target_id=aurora.id),
    Relation(label="SERVES_ON_BOARD_OF", source_id=julian.id, target_id=aether.id),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=aurora.id, properties={"amount": "$30M", "equity": "15%"}),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=synapse.id, properties={"amount": "$60M", "equity": "22%"}),
    Relation(label="JOINT_VENTURE_WITH", source_id=aurora.id, target_id=aether.id, properties={"partner": "Helios Capital"}),
    Relation(label="INVESTED_IN", source_id=helios.id, target_id=aether.id, properties={"amount": "$45M"}),
    Relation(label="LICENSES_TO", source_id=aurora.id, target_id=synapse.id, properties={"technology": "Optical Communication Protocols"}),
    Relation(label="FUNDED", source_id=helios.id, target_id=nexus.id, properties={"mode": "Covert"}),
    Relation(label="TESTBED_FOR", source_id=synapse.id, target_id=nexus.id),
]

# 11.4
manual_store.upsert_nodes(nodes)
manual_store.upsert_relations(relations)

# 11.5
index_manual = PropertyGraphIndex.from_existing(
    property_graph_store=manual_store,
    llm=llm,
    embed_model=embed_model,
    use_async=False,
)

# 11.5.1
print(f"Manual graph created with {len(nodes)} nodes and {len(relations)} relationships!")

In [ ]:
# 12.0 Query the manually constructed graph
manual_query_engine = index_manual.as_query_engine(include_text=False, use_async=False)

# 12.1
response_manual = manual_query_engine.query(
    "How is Dr. Elena Vance connected to Synapse Dynamics and Project Nexus through Helios Capital?"
)

# 12.2
print("=== 💡 MANUAL GRAPHRAG RESPONSE ===")
print(response_manual.response)